In [0]:


from pyspark.sql.functions import *

print("Starting incremental transformation...")


# CHECKPOINT LOCATION

checkpoint_path = "abfss://letssayapi@jobdata.dfs.core.windows.net/checkpoints/ramen_transformation"


# READ BRONZE TABLE AS STREAM

bronze_stream = (
    spark.readStream
    .table("`for-job-prac-lms`.default.bronze_ramen_reviews")
)


transformed_df = (
    bronze_stream

    # Remove unrated rows
    .filter(col("stars") != "Unrated")

    # Convert rating to numeric
    .withColumn("stars", col("stars").cast("double"))

    # Remove invalid numeric rows
    .filter(col("stars").isNotNull())

    # Remove rows with important nulls
    .dropna(subset=["brand", "country", "style"])
)


transformed_df = transformed_df.withColumn(
    "rating_category",
    when(col("stars") >= 4, "Excellent")
    .when(col("stars") >= 3, "Good")
    .otherwise("Average")
)

transformed_df = transformed_df.withColumn(
    "premium_ramen",
    when(col("stars") >= 4.5, "Yes")
    .otherwise("No")
)

# WRITE TO SILVER TABLE


query = (
    transformed_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("`for-job-prac-lms`.default.silver_ramen_reviews")
)

query.awaitTermination()

print("Silver incremental transformation completed successfully")

Starting incremental transformation...
Silver incremental transformation completed successfully
